## **Problem Statement**

### Business Context[¶](#Business-Context)

NewsFindr is redefining news discovery by delivering real-time news updates tailored to user interests. Traditional search methods and generic news feeds often lead to information overload and inefficiencies, making it challenging for users to access relevant and trustworthy content efficiently.

To address this, NewsFindr wants to leverage Agentic AI to build an AI-powered news retrieval agent that ensures accuracy and credibility. By utilizing a structured, multi-step approach, the system will provide secure, fair, and explainable recommendations - enhancing user engagement, optimizing content discovery, and improving access to timely and relevant news.

### Objective[¶](#Objective)

- Provide real-time, personalized news retrieval to help users discover relevant
  content effortlessly.
- Ensure accuracy and credibility by sourcing news from trusted platforms and minimizing misinformation.
- Improve user engagement through seamless content discovery, reducing information overload.
- Streamline the news consumption process by eliminating outdated and irrelevant content, providing a refined reading experience.

## **Please read the instructions carefully before starting the project**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.

- Blanks '\_\_\_\_\_' are provided in the notebook that
  needs to be filled with an appropriate code to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
- Identify the task to be performed correctly, and only then proceed to write the required code.
- Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
- Add the results/observations at the end of the analysis and submit the same.

## **Installing and Importing Necessary Libraries and Dependencies**

In [1]:
#Install all the libraries
%pip install -qU langchain-groq

!pip install -q langchain \
                langchainhub==0.1.20 \
                langchain-experimental==0.0.62\
                langchain_huggingface\
                langchain-groq

!pip install -qU duckduckgo-search
!pip uninstall -y duckduckgo_search
!pip install ddgs

!pip install numpy==1.26


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.7/202.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires nu

**Note:**

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [1]:
import json
import os
import pandas as pd
import sqlite3

from langchain.agents import create_sql_agent, initialize_agent # Added initialize_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool

from pydantic import BaseModel, Field, ValidationError

from ddgs import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [2]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')#Complete the code by calling the Groq API key


Initializes Gorq model with different temperature settings.

In [3]:
#from google.colab import userdata
from langchain_groq import ChatGroq  # Import Groq LLM

# Get the API key from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY') #Complete the code by calling the API key

# High creativity LLM
llm_high = ChatGroq(
    model="llama-3.3-70b-versatile", #Call the Groq model
    temperature=0.9, #Complete the code by providing high temperature
    groq_api_key=groq_api_key)

# Low creativity (deterministic) LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile", #Call the Groq model
    temperature=0.1, #Complete the code by providing low temperature
    groq_api_key=groq_api_key
)


## **Checking LLMs response**

A low temperature setting ensures consistent and repeatable responses, making the output more deterministic. It is ideal for tasks requiring accuracy, such as fact-based queries, structured data extraction, and rule-based decision-making.

In [ ]:
for i in range(5):
  response = llm.predict("What is Artificial Intelligence?") #Complete the code by asking a simple question to the llm
  print(response)


**Artificial Intelligence (AI)** is a field of computer science that focuses on creating intelligent machines capable of performing tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, experiences, and interactions, enabling them to improve their performance over time.
2. **Reasoning**: AI systems can draw inferences, make decisions, and solve problems using logical rules and algorithms.
3. **Problem-solving**: AI systems can identify and solve complex problems, often using a combination of machine learning, optimization, and search techniques.
4. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text, allowing them to perceive and respond to their environment.

The primary goals of AI research are to:

1. **Mimic human intelligence**: Create machines that can think, learn, and behave like humans.
2. **Augment human capabilities**: Develop AI systems that can assist humans in var

Observation: The model generated highly consistent responses across multiple executions. Since a low temperature value was used, the output remained stable with minimal variation in wording and content. This behavior is desirable for tasks requiring accuracy, reproducibility, and deterministic decision-making.

In [ ]:
for i in range(5):
  response = llm_high.predict("What is Artificial Intelligence?") #Complete the code by asking a simple question to the llm
  print(response)


**Artificial Intelligence (AI) Definition:**
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

* Learning
* Problem-solving
* Reasoning
* Perception
* Understanding language

**Key Characteristics of AI:**

1. **Machine Learning**: AI systems can learn from data and improve their performance over time.
2. **Autonomy**: AI systems can operate independently, making decisions without human intervention.
3. **Adaptability**: AI systems can adapt to new situations and learn from experience.
4. **Reasoning and Problem-Solving**: AI systems can analyze data, identify patterns, and make decisions based on that analysis.

**Types of AI:**

1. **Narrow or Weak AI**: Designed to perform a specific task, such as facial recognition, language translation, or playing chess.
2. **General or Strong AI**: A hypothetical AI system that possesses human-like intelligence and can perform any intellectu

The high-temperature model generated responses with greater diversity and creativity across multiple executions. While the core explanation of Artificial Intelligence remained consistent, the structure, examples, applications, benefits, and challenges varied between responses. This demonstrates that higher temperature increases randomness and response variability, making it useful for tasks such as brainstorming, query generation, content creation, and idea exploration.

## **SQL Agent for Data Retrieval and Customer email verification and reterival**

Facilitates efficient data retrieval and modification, enabling seamless customer information management through SQL interactions

**Function setup\_sql\_agent**: Initializes an AI-powered SQL agent for querying and updating a database.
It connects to an SQL database, and sets up an agent and check for the presence of the email in the DB, if not found return as "email not found", if email found return with the "name of the customer and area of interest".

In [4]:
from google.colab import files

uploaded = files.upload()


Saving customer.db to customer.db


In [5]:
db = SQLDatabase.from_uri("sqlite:///customer.db")                                             # complete the code by loading the SQLite database

#Initialize the system message
system_message = """
You are a SQL assistant.
Use the available database tables to answer user questions accurately.
Return only relevant information.
"""                                                       #Complete the coe by providing the system_message
#Initialize the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

#Create the SQL agent with the system message
db_agent = create_sql_agent(
    llm=llm,                                                                 #Complete the code by calling the llm
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)


In [ ]:
query = "Fetch all unique email addresses from the customers table"

output = db_agent.invoke(query)

output


{'input': 'Fetch all unique email addresses from the customers table',
 'output': "[('alice.6eb33c45-5@gmail.com',), ('alice.d56e53b6-1@gmail.com',), ('emma.a88fec03-c@gmail.com',), ('george.5483cb53-8@gmail.com',), ('hannah.4770b814-2@gmail.com',), ('ian.203631a0-b@gmail.com',), ('ian.aee37571-7@gmail.com',), ('julia.b5a14512-c@gmail.com',), ('julia.cf4d1edb-9@gmail.com',), ('julia.d77d96f3-3@gmail.com',)]"}

The SQL agent correctly translated the user query into SQL and returned all unique email addresses from the database, validating the successful integration of the LLM with the SQLite database.

In [ ]:
email="nothere@gmail.com"                  #Complete the code by providing the email id which is not present in the database to check the response of LLm
query= f"Fetch all records with email_id {email}"

output=db_agent.invoke(query)

output


{'input': 'Fetch all records with email_id nothere@gmail.com',
 'output': 'There are no records with email "nothere@gmail.com" in the database.'}

The agent successfully searched the database and correctly reported that the specified email address does not exist, demonstrating reliable query execution and error handling.

In [ ]:
email1="alice.6eb33c45-5@gmail.com"                  #Complete the code by providing the email id which is present in the database to check the response of LLm

query= f"Fetch all the Interests with email_id {email1} in a list"

output=db_agent.invoke(query)

output


{'input': 'Fetch all the Interests with email_id alice.6eb33c45-5@gmail.com in a list',
 'output': '["Politics", "Technology", "Business"]'}

## **Generating Expanded Search Queries for Current News**

Generating Expanded Search Queries for Current News
This function, expand\_search\_queries, generates precise and up-to-date search queries based on the user's interests. It ensures that each query is crafted to retrieve breaking news, trending topics, or recent developments without relying on specific years.

The function takes user interests and User\_query as inputs and processes through an LLM to generate a relevant search query.

A system prompt guides the model to focus only on current events and avoid outdated information.

The expanded search queries are stored in expanded\_search and returned for later use in retrieving news articles.

In [6]:
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    # Handle dict input
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query","")
    else:
        # Handle string input fallback
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""

    system_prompt = """
You are a news query generation expert.

Generate one concise, time-sensitive search query suitable for finding recent news.

Rules:
1. Focus on current events.
2. Keep query short.
3. Include important keywords.
4. Return only the query.
"""                                                             #Complete the code by providing a clear instructions in the system prompt

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries

# Define the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)


## **Fetch News Results Using DuckDuckGo**

This function retrieves news results from DuckDuckGo based on the user's expanded search queries.

In [7]:
def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):                              #Complete the code by providing the max_result value
            title = r.get("title", "")
            url = r.get("href", "")
            body = r.get("body", "")
            res={"title":title,"url":url,"body":body}
            results.append(res)
    return results
ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)


## **Filter Relevant and Trustworthy URLs Based on User Interests**

This function filters search result URLs, assesses the credibility of news sources, and retains only those that are relevant to the user's interests.

Extracts URLs and their body content from the search results.

Updates the state with the refined list of relevant URLs and their descriptions.

In [8]:
def filter_with_llm(search_results: list) -> list:

    search_results= json.dumps(search_results, indent=2)
    system_prompt = """
You are a news credibility evaluator.

Analyze search results and keep only results from trustworthy and reputable sources.

Prefer:
- Reuters
- BBC
- CNBC
- Bloomberg
- Financial Times
- Associated Press
- The Hindu
- Economic Times

Return only credible URLs.
"""                                              #Complete the code by providing a clear instructions in the system prompt

    prompt =  "Evaluate the credibility of the following search results and return only trustworthy URLs:\n" + search_results                              #Complete the code by writing a prompt to evaluate the credibility of the URLs

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content


credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)


## **Generate summary for the URLs**

This function generates summaries for the credible URLs

Returns the URLs and the Summary based on the users interest

In [9]:
def summarize_news(url_list) -> dict:
    """
    Summarizes key news points from a list of URLs and returns both the summary and the URLs.
    Since only URLs are available, the LLM will generate a high-level summary
    based on the website context or domain.
    """

    system_prompt = """
You are a professional news analyst.

Generate a concise summary of the major news themes represented by the URLs.

Focus on:
- Important developments
- Trends
- Business implications
- Technology implications

Keep summary factual.
"""                                             #Complete the code by providing a clear instructions in the system prompt

    # Pass the URLs to the LLM for summarization
    prompt = "Summarize the news represented by these URLs:\n" + "\n".join(url_list)                                   #Complete the code by writing a prompt to summarize the URLs

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )

    summary_text = response.content

    # Return a dictionary containing the summary and the original list of URLs
    return {"summary": summary_text, "urls": url_list}


summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)


## **Creating an Agent**

In [10]:
#Final code
tools = [expand_tool,ddg_search_tool,credibility_tool,summarize_tool,]                                                    #Complete the code by calling the already defined function
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


## **Display Final Verified URLs and Summary based on the User interest and User query**

In [ ]:
#Final code
email = "alice.6eb33c45-5@gmail.com"                                                             #Complete the code by providing a user email
interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

# Normalize to list
if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
    interests = interest_result["output"]["items"]
elif isinstance(interest_result["output"], str):
    interests = [i.strip() for i in interest_result["output"].split(",")]
else:
    interests = interest_result["output"]

# Define the user query
user_query = "latest technology and business news"                                                       #Complete the code by writing the relevant User_query

# Agent prompt stays the same, just note we’re passing the dict
agent_prompt = f"""
User Interests: {interests}

User Query: {user_query}

Perform the following:
1. Expand interests into search queries.
2. Search for recent news.
3. Filter credible sources.
4. Summarize findings.
5. Return URLs and summary.
"""                                             #Complete the code by providing the agent prompt
response = agent.run(agent_prompt)                                                  #Complete the code by calling the prompt agent
print("\n======= FINAL RESPONSE =======")
print(response)


Thought: The user has provided their interests as "Politics", "Technology", and "Business", and they want to know the latest news about "Technology" and "Business". To provide them with relevant and recent news, I need to first expand their interests into precise, time-sensitive news search queries.

Action: ExpandSearchQueries
Action Input: ['["Politics"', '"Technology"', '"Business"]'], "latest technology and business news"
Observation: ['"latest political news updates"', '"latest tech advancements 2024"', '"latest business news today"', '"latest tech industry updates"']
Thought:Now that I have the expanded search queries, I need to search for recent news using these queries. However, the user's query specifically mentions "latest technology and business news", so I should prioritize the queries related to technology and business.

Action: DuckDuckGoSearch
Action Input: '"latest tech advancements 2024"'
Observation: [{'title': 'Scale Ai Co Founder | TikTok', 'url': 'https://www.tikto

The agent successfully executed a multi-step reasoning workflow by sequentially invoking four tools: query expansion, news search, credibility filtering, and news summarization. The agent automatically selected the appropriate tools, processed intermediate outputs, and generated a final response containing relevant technology and business news from credible sources. This demonstrates the effectiveness of LangChain agents in orchestrating multiple tools to solve complex tasks autonomously.

## **Retrieve URLs and Summaries for Three Areas of Interest**

In [12]:
def query_response(email: str, user_query: str) -> str:
    # Fetch interests based on the provided email
    interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

    # Normalize to list
    if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
      interests = interest_result["output"]["items"]
    elif isinstance(interest_result["output"], str):
        interests = [i.strip() for i in interest_result["output"].split(",")]
    else:
       interests = interest_result["output"]


    # Agent prompt stays the same, just note we’re passing the dict
    agent_prompt = f"""User Interests: {interests}

User Query: {user_query}

IMPORTANT:
The User Query has higher priority than User Interests.

If the query is specific, focus primarily on the query.
Use user interests only when they directly support the query.
Do not search unrelated interest categories.

Perform the following:
1. Expand the query into search terms.
2. Search recent news.
3. Filter credible sources.
4. Summarize findings.
5. Return URLs and summary.
"""
    response = agent.run(agent_prompt)
    print("\n======= FINAL RESPONSE =======")
    print(response)

    return response


In [ ]:
#Check for the user ID
email = "alice.6eb33c45-5@gmail.com"
user_query = "latest political developments in India"                                                      #Complete the code by providing the relevant user query
response = query_response(email, user_query)


Thought: The user query is "latest political developments in India", which is a specific query. The user interests are ['Politics', 'Technology', 'Business'], but since the query is specific, we will focus primarily on the query and use the user interests only if they directly support the query. In this case, 'Politics' is directly related to the query. We need to expand the query into precise, time-sensitive news search queries.

Action: ExpandSearchQueries
Action Input: ['latest political developments in India']
Observation: ['"India latest political news today"']
Thought:Now that we have the expanded search query, we can use it to search for recent news on DuckDuckGo.

Action: DuckDuckGoSearch
Action Input: '"India latest political news today"'
Observation: [{'title': 'India | Latest News & Updates | BBC News', 'url': 'https://www.bbc.com/news/world/asia/india', 'body': "8 hours ago - Former President Zuma's trip to India reignites anger over a decade-old corruption scandal."}, {'ti

In [18]:
print("\n========== SYSTEM METRICS ==========")
print("Number of Interests: 3")
print("User Query: latest political developments in India")
print("Pipeline Status: SUCCESS")



========== SYSTEM METRICS ==========
Number of Interests: 3
User Query: latest political developments in India
Pipeline Status: SUCCESS


The NewsFindr agent successfully retrieved the user's profile information, identified relevant interests, and prioritized the user query to generate focused search terms. The agent then searched for recent political news related to India, filtered the results using credibility assessment, and summarized information from trusted sources. The output demonstrates effective integration of user profiling, query expansion, information retrieval, source credibility evaluation, and automated news summarization. The modification to prioritize the user query improved the relevance of the retrieved news by ensuring that search results were focused on Indian political developments rather than unrelated interest categories.

In [ ]:
#Check for the user ID
email = "emma.a88fec03-c@gmail.com"
user_query = "latest technology innovations and AI news"                                                     #Complete the code by providing the relevant user query
response = query_response(email, user_query)


Thought: The user query is "latest technology innovations and AI news", which is quite specific. It seems to be focused on technology, particularly AI. Since the query has higher priority, I should focus on expanding this query into precise search terms, rather than relying on the user interests. However, the user interest in "Technology" does support the query, so I can consider that as well. I will start by expanding the query into search terms.

Action: ExpandSearchQueries
Action Input: ["latest technology innovations and AI news"]
Observation: ['"AI breakthroughs 2024"']
Thought:The observation from the ExpandSearchQueries action is a list containing the search term '"AI breakthroughs 2024"'. This term is a precise and time-sensitive search query that can be used to find relevant news articles about the latest AI innovations.

Action: DuckDuckGoSearch
Action Input: "AI breakthroughs 2024"
Observation: [{'title': 'AI boom - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/AI_boom',

In [19]:
print("\n========== SYSTEM METRICS ==========")
print("Number of Interests: 2")
print("User Query: latest technology innovations and AI news")
print("Pipeline Status: SUCCESS")



========== SYSTEM METRICS ==========
Number of Interests: 2
User Query: latest technology innovations and AI news
Pipeline Status: SUCCESS


In [13]:
#Check for the user ID
email = "julia.cf4d1edb-9@gmail.com"
user_query = "latest international sports news"                                                      #Complete the code by providing the relevant user query
response = query_response(email, user_query)


Thought: The user query is "latest international sports news", which is a specific query. The user interests include "Sports", "Entertainment", and "International", but since the query is specific, we will focus primarily on the query and use the user interests only if they directly support the query. In this case, "International" and "Sports" are directly relevant to the query. We need to expand the query into precise search terms to get relevant news articles.

Action: ExpandSearchQueries
Action Input: ["latest international sports news"]
Observation: ['"international sports news today"']
Thought:Thought: Now that we have the expanded search query, we can use it to search for recent news articles. The expanded query is "international sports news today", which is a precise and time-sensitive search term. We will use this query to search for the top 5 recent news articles on DuckDuckGo.

Action: DuckDuckGoSearch
Action Input: "international sports news today"
Observation: [{'title': 'S

In [20]:
print("\n========== SYSTEM METRICS ==========")
print("Number of Interests: 3")
print("User Query: latest international sports news")
print("Pipeline Status: SUCCESS")



========== SYSTEM METRICS ==========
Number of Interests: 3
User Query: latest international sports news
Pipeline Status: SUCCESS


The NewsFindr agent successfully expanded the user query into a focused and time-sensitive search term, retrieved recent international sports news using DuckDuckGo, and filtered the results based on source credibility. The system identified Reuters, BBC, and Times of India as reliable news sources and generated a consolidated summary of current sports-related developments. The workflow demonstrates effective integration of query expansion, news retrieval, credibility assessment, and automated summarization to deliver relevant information aligned with the user's interests.

# Conclusion[¶](#Conclusion)

In this notebook, an AI-powered personalized news retrieval system was successfully developed using Large Language Models, SQL-based customer data retrieval, and web search integration. The system validates customer information from a database, retrieves user interests through an SQL agent, expands interests into relevant and time-sensitive search queries, gathers recent news using DuckDuckGo Search, filters results based on source credibility, and generates concise summaries of the retrieved information.

The workflow demonstrates how LLMs can be combined with structured databases and external information sources to create intelligent, context-aware applications. Through multiple test cases covering politics, technology, artificial intelligence, and international sports, the agent consistently generated relevant search queries, retrieved credible information, and produced meaningful summaries aligned with user interests and queries.

This project highlights the practical use of agent-based AI systems for personalized information retrieval and showcases the integration of LLM reasoning, database querying, tool usage, and automated summarization within a unified workflow.